# Reproduce DeepSequence paper findings

Fast path: **load locked JSON artifacts** under `ab_runs/` and rebuild key tables/figures.
Optional long cells re-run Direct-MH bake-offs (hours on full 800).

## Environment
- Python: `.venv-test`
- `TF_USE_LEGACY_KERAS=1` (required for this stack)
- Enterprise daily panel: `DEEPSEQUENCE_DATA_DIR` → Jubilant `data/` (not shipped)
- Package imports only — no `examples/*.py` library modules


In [ ]:
import os
from pathlib import Path

# Prefer legacy Keras before TF imports elsewhere
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

ROOT = Path.cwd()
if not (ROOT / "PAPER.md").exists():
    # notebook may be opened with cwd = examples/
    if (ROOT.parent / "PAPER.md").exists():
        ROOT = ROOT.parent
        os.chdir(ROOT)

print("repo", ROOT)
print("TF_USE_LEGACY_KERAS", os.environ.get("TF_USE_LEGACY_KERAS"))
print("DEEPSEQUENCE_DATA_DIR", os.environ.get("DEEPSEQUENCE_DATA_DIR", "<unset>"))


## Load artifact helpers (package)

`deepsequence_hierarchical_attention.eval.paper_artifacts` reads locked JSON and builds like-for-like Direct-MH tables.


In [ ]:
from deepsequence_hierarchical_attention.eval.paper_artifacts import (
    DEFAULT_PATHS,
    bakeoff_table,
    format_markdown_table,
    like_for_like_direct_table,
    load_json,
    zero_rate_summary,
)

zr = zero_rate_summary()
zr


## Zero rate: daily vs weekly (Table Z)


In [ ]:
import pandas as pd
pd.DataFrame([zr])


## Weekly Direct-MH (Table W)

Artifact: `ab_runs/weekly/weekly_mh8_locked800_s42.json`


In [ ]:
weekly = load_json(DEFAULT_PATHS["weekly_mh"])
rows = bakeoff_table(weekly, horizons=["1", "4", "8"])
print(format_markdown_table(rows, ["model", "method", "h1", "h4", "h8"]))
pd.DataFrame(rows)


In [ ]:
rows_c = bakeoff_table(weekly, horizons=["1", "4", "8"], cum=True)
print("CumMAE")
print(format_markdown_table(rows_c, ["model", "method", "h1", "h4", "h8"]))
pd.DataFrame(rows_c)


## Daily Direct-MH (Table D) — like-for-like protocol

Artifact: `ab_runs/weekly/daily_direct_mh60_locked800_s42.json`

Recursive daily Table 1 (`ab_runs/reclaim/daily_mh_1_60_level1_cross_off_all_models.json`) remains the **primary** portfolio claim.


In [ ]:
daily = load_json(DEFAULT_PATHS["daily_direct_mh"])
hs = ["1", "7", "14", "28", "56", "60"]
rows = bakeoff_table(daily, horizons=hs)
print(format_markdown_table(rows, ["model", "method"] + [f"h{h}" for h in hs]))
pd.DataFrame(rows)


In [ ]:
rows_c = bakeoff_table(daily, horizons=hs, cum=True)
print("CumMAE")
pd.DataFrame(rows_c)


## Like-for-like Direct↔Direct (Table L)

Matched leads: weekly \(h=1/4/8\) ≈ daily \(h=7/28/56\). Absolute IWMAE is **not** cross-grain comparable.


In [ ]:
ll = like_for_like_direct_table(weekly, daily)
pd.DataFrame(ll)


## Recursive daily reference (primary Table 1)

Kept labeled **recursive**; do not mix with Direct-MH rankings above.


In [ ]:
rec = load_json(DEFAULT_PATHS["daily_recursive_mh"])
models = ["deepsequence", "temporal_transformer", "lightgbm"]
horizons = ["1", "7", "14", "28", "60"]
out = []
for m in models:
    row = {"model": m, "method": rec["models"][m].get("method", "recursive")}
    for h in horizons:
        block = rec["models"][m]["by_horizon"][h]
        if "overall" in block:
            block = block["overall"]
        row[f"h{h}"] = block.get("iwmae_rounded")
    out.append(row)
pd.DataFrame(out)


## Multi-seed daily IWMAE summary (Figure 6 source)


In [ ]:
ms = load_json(DEFAULT_PATHS["daily_multiseed"])
block = ms["iwmae_mean_std"]
rows = []
for h in ["1", "7", "14", "28", "60"]:
    for m in ["deepsequence", "temporal_transformer", "lightgbm"]:
        e = block[h][m]
        rows.append({"h": int(h), "model": m, "mean": e["mean"], "std": e["std"]})
pd.DataFrame(rows).pivot(index="h", columns="model", values="mean")


## Regenerate Direct-MH comparison figures

Writes `paper_figures/fig_zero_rate_daily_vs_weekly.png`, `fig_weekly_daily_direct_iwmae.png`, `fig_weekly_daily_direct_cummae.png`.


In [ ]:
%run paper_figures/make_weekly_daily_direct_compare.py
from IPython.display import Image, display
for stem in [
    "fig_zero_rate_daily_vs_weekly",
    "fig_weekly_daily_direct_iwmae",
    "fig_weekly_daily_direct_cummae",
]:
    display(Image(filename=f"paper_figures/{stem}.png"))


## Spike / holiday qualitative pointers

Qualitative forecast dumps (planning-rate vs spikes; holiday response) live under `paper_figures/`:

- `fig_forecast_daily_onestep.png` / `fig_forecast_daily_recursive.png`
- Binary / country holiday variants: `fig_forecast_daily_*_hol_*.png`
- Spike diagnostics: `fig_spike_diag_panel.png`

See `PAPER.md` §5.6–5.8 and §6.1. These are **not** regenerated here (require prediction dumps).


## Optional: re-run Direct-MH bake-offs (long)

Default is **off**. Set `RUN_RERUN = True` only with Jubilant data + GPU/CPU budget.


In [ ]:
RUN_RERUN = False  # flip to True to re-train Direct-MH

if RUN_RERUN:
    import subprocess, shlex
    data = os.environ.get("DEEPSEQUENCE_DATA_DIR")
    if not data:
        raise RuntimeError("Set DEEPSEQUENCE_DATA_DIR")
    cmds = [
        f'''TF_USE_LEGACY_KERAS=1 .venv-test/bin/python -m deepsequence_hierarchical_attention.eval.weekly_mh '''
        f'''--data_dir ab_runs/weekly/panel_locked800 --feature_config feature_config_weekly.yaml '''
        f'''--sku_list ab_runs/recompare/sku_list_daily_data42.json --max_skus 800 '''
        f'''--horizon 8 --report_horizons 1,4,8 --models deepsequence,tsb,lightgbm '''
        f'''--epochs 15 --seed 42 --out_json ab_runs/weekly/weekly_mh8_locked800_s42.json''',
        f'''TF_USE_LEGACY_KERAS=1 .venv-test/bin/python -m deepsequence_hierarchical_attention.eval.weekly_mh '''
        f'''--data_dir {shlex.quote(data)} --feature_config feature_config.yaml '''
        f'''--dataset daily_direct_mh --sku_list ab_runs/recompare/sku_list_daily_data42.json '''
        f'''--max_skus 800 --horizon 60 --report_horizons 1,7,14,28,56,60 --mase_season 7 '''
        f'''--models deepsequence,tsb,lightgbm --epochs 15 --seed 42 '''
        f'''--out_json ab_runs/weekly/daily_direct_mh60_locked800_s42.json''',
    ]
    for c in cmds:
        print("RUN:", c)
        subprocess.check_call(c, shell=True)
else:
    print("Skipping re-run (RUN_RERUN=False). Artifacts already loaded above.")
